In [ ]:
import optuna
from sklearn.datasets import load_diabetes
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [8]:
# We will load the actual diabetes dataset from an external source
import pandas as pd

# Load the Pima Indian Diabetes dataset (from UCI repository)
url = "https://raw.githubusercontent.com/jbrownlee/Datasets/master/pima-indians-diabetes.data.csv"
columns = ['Pregnancies', 'Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI',
           'DiabetesPedigreeFunction', 'Age', 'Outcome']

# Load the dataset
df = pd.read_csv(url, names=columns)

df.head()

,Pregnancies,Glucose,BloodPressure,SkinThickness,Insulin,BMI,DiabetesPedigreeFunction,Age,Outcome
0,6,148,72,35,0,33.6,0.627,50,1
1,1,85,66,29,0,26.6,0.351,31,0
2,8,183,64,0,0,23.3,0.672,32,1
3,1,89,66,23,94,28.1,0.167,21,0
4,0,137,40,35,168,43.1,2.288,33,1


In [9]:
df.isnull().sum()

Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64

In [10]:
import numpy as np

# Replace zero values with NaN in columns where zero is not a valid value
cols_with_missing_vals = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
df[cols_with_missing_vals] = df[cols_with_missing_vals].replace(0, np.nan)

# Impute the missing values with the mean of the respective column
df.fillna(df.mean(), inplace=True)

# Check if there are any remaining missing values
print(df.isnull().sum())


Pregnancies                 0
Glucose                     0
BloodPressure               0
SkinThickness               0
Insulin                     0
BMI                         0
DiabetesPedigreeFunction    0
Age                         0
Outcome                     0
dtype: int64


In [11]:
X = df.drop('Outcome', axis=1)
y = df['Outcome']
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [12]:
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape: {X_test.shape}")

X_train shape: (614, 8)
X_test shape: (154, 8)


In [13]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

In [ ]:
# Define the objective function for Optuna
def objective(trial):
    # Suggest hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 1, 50)
    min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
    
    # Create the model with the suggested hyperparameters
    model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth,
                                   min_samples_split=min_samples_split, random_state=42)
    
    # Perform cross-validation and return the mean accuracy
    score = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    return score.mean()

In [15]:
# Create a study object and optimize the objective function
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())
# Optimize the objective function
study.optimize(objective, n_trials=50)

[I 2025-03-30 02:48:15,003] A new study created in memory with name: no-name-eca5a5e3-b2d5-4315-9264-c28e1d22838b
[I 2025-03-30 02:48:16,093] Trial 0 finished with value: 0.7638544582167134 and parameters: {'n_estimators': 109, 'max_depth': 9, 'min_samples_split': 3}. Best is trial 0 with value: 0.7638544582167134.
[I 2025-03-30 02:48:17,171] Trial 1 finished with value: 0.7704118352658936 and parameters: {'n_estimators': 107, 'max_depth': 11, 'min_samples_split': 8}. Best is trial 1 with value: 0.7704118352658936.
[I 2025-03-30 02:48:18,654] Trial 2 finished with value: 0.7720245235239238 and parameters: {'n_estimators': 142, 'max_depth': 32, 'min_samples_split': 4}. Best is trial 2 with value: 0.7720245235239238.
[I 2025-03-30 02:48:20,611] Trial 3 finished with value: 0.7703985072637611 and parameters: {'n_estimators': 183, 'max_depth': 22, 'min_samples_split': 9}. Best is trial 2 with value: 0.7720245235239238.
[I 2025-03-30 02:48:22,525] Trial 4 finished with value: 0.767159802745

In [16]:
print("Best hyperparameters: ", study.best_params)
print("Best accuracy: ", study.best_value)

Best hyperparameters:  {'n_estimators': 185, 'max_depth': 34, 'min_samples_split': 12}
Best accuracy:  0.7801679328268692


In [17]:
from sklearn.metrics import accuracy_score
# Train the model with the best hyperparameters
best_model = RandomForestClassifier(**study.best_params, random_state=42)
best_model.fit(X_train, y_train)
# Make predictions on the test set
y_pred = best_model.predict(X_test)
# Calculate the accuracy
accuracy = accuracy_score(y_test, y_pred)
print(f"Test set accuracy: {accuracy:.4f}")

Test set accuracy: 0.7662


In [ ]:
# more sampler
# https://optuna.readthedocs.io/en/stable/reference/samplers.html
# https://optuna.readthedocs.io/en/stable/tutorial/10_key_features/002_configurations.html

# study = optuna.create_study(direction="maximize", sampler=optuna.samplers.RandomSampler())
# study = optuna.create_study(direction="maximize", sampler=optuna.samplers.GridSampler())
# study = optuna.create_study(direction="maximize", sampler=optuna.samplers.CmaEsSampler())
# study = optuna.create_study(direction="maximize", sampler=optuna.samplers.NSGAIISampler())

In [18]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import cross_val_score

# Define the objective function
def objective(trial):
    # Suggest values for the hyperparameters
    n_estimators = trial.suggest_int('n_estimators', 50, 200)
    max_depth = trial.suggest_int('max_depth', 3, 20)

    # Create the RandomForestClassifier with suggested hyperparameters
    model = RandomForestClassifier(
        n_estimators=n_estimators,
        max_depth=max_depth,
        random_state=42
    )

    # Perform 3-fold cross-validation and calculate accuracy
    score = cross_val_score(model, X_train, y_train, cv=3, scoring='accuracy').mean()

    return score  # Return the accuracy score for Optuna to maximize


In [19]:
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.RandomSampler())  # We aim to maximize accuracy
study.optimize(objective, n_trials=50)  # Run 50 trials to find the best hyperparameters

[I 2025-03-30 03:05:29,397] A new study created in memory with name: no-name-3b4c63ed-ace3-41b2-b49b-a988f77ca882
[I 2025-03-30 03:05:29,945] Trial 0 finished with value: 0.7736091184441256 and parameters: {'n_estimators': 89, 'max_depth': 20}. Best is trial 0 with value: 0.7736091184441256.
[I 2025-03-30 03:05:30,302] Trial 1 finished with value: 0.7736091184441256 and parameters: {'n_estimators': 71, 'max_depth': 6}. Best is trial 0 with value: 0.7736091184441256.
[I 2025-03-30 03:05:30,805] Trial 2 finished with value: 0.7556910569105691 and parameters: {'n_estimators': 117, 'max_depth': 4}. Best is trial 0 with value: 0.7736091184441256.
[I 2025-03-30 03:05:31,355] Trial 3 finished with value: 0.7768770923003347 and parameters: {'n_estimators': 110, 'max_depth': 18}. Best is trial 3 with value: 0.7768770923003347.
[I 2025-03-30 03:05:31,927] Trial 4 finished with value: 0.7768850629682768 and parameters: {'n_estimators': 112, 'max_depth': 17}. Best is trial 4 with value: 0.77688506

In [20]:
# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7850231149370317
Best hyperparameters: {'n_estimators': 116, 'max_depth': 14}


In [21]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')

Test Accuracy with best hyperparameters: 0.76


In [22]:
search_space = {
    'n_estimators': [50, 100, 150, 200],
    'max_depth': [5, 10, 15, 20]
}

In [23]:
# Create a study and optimize it using GridSampler
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.GridSampler(search_space))
study.optimize(objective)

[I 2025-03-30 03:06:49,189] A new study created in memory with name: no-name-e3fbdfe4-d3d5-417b-b474-02f156408348
[I 2025-03-30 03:06:49,722] Trial 0 finished with value: 0.7654391838036028 and parameters: {'n_estimators': 100, 'max_depth': 5}. Best is trial 0 with value: 0.7654391838036028.
[I 2025-03-30 03:06:50,486] Trial 1 finished with value: 0.7735772357723577 and parameters: {'n_estimators': 150, 'max_depth': 10}. Best is trial 1 with value: 0.7735772357723577.
[I 2025-03-30 03:06:50,798] Trial 2 finished with value: 0.7687151283277539 and parameters: {'n_estimators': 50, 'max_depth': 15}. Best is trial 1 with value: 0.7735772357723577.
[I 2025-03-30 03:06:51,309] Trial 3 finished with value: 0.7752351347042882 and parameters: {'n_estimators': 100, 'max_depth': 15}. Best is trial 3 with value: 0.7752351347042882.
[I 2025-03-30 03:06:51,809] Trial 4 finished with value: 0.7703491152558585 and parameters: {'n_estimators': 100, 'max_depth': 20}. Best is trial 3 with value: 0.775235

In [24]:
# Print the best result
print(f'Best trial accuracy: {study.best_trial.value}')
print(f'Best hyperparameters: {study.best_trial.params}')

Best trial accuracy: 0.7817391997449387
Best hyperparameters: {'n_estimators': 50, 'max_depth': 10}


In [25]:
from sklearn.metrics import accuracy_score

# Train a RandomForestClassifier using the best hyperparameters from Optuna
best_model = RandomForestClassifier(**study.best_trial.params, random_state=42)

# Fit the model to the training data
best_model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = best_model.predict(X_test)

# Calculate the accuracy on the test set
test_accuracy = accuracy_score(y_test, y_pred)

# Print the test accuracy
print(f'Test Accuracy with best hyperparameters: {test_accuracy:.2f}')

Test Accuracy with best hyperparameters: 0.75


## Optuna Visualization

In [26]:
# For visualizations
from optuna.visualization import plot_optimization_history, plot_parallel_coordinate, plot_slice, plot_contour, plot_param_importances

In [27]:
# 1. Optimization History
plot_optimization_history(study)

In [28]:
# 2. Parallel Coordinate Plot
plot_parallel_coordinate(study)

In [29]:
# 3. Slice Plot
plot_slice(study)

In [30]:
# 4. Contour Plot
plot_contour(study)

In [31]:
# 5. Parameter Importances
plot_param_importances(study)

## Optimizing ML Models

In [36]:
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

<frozen importlib._bootstrap>:488: RuntimeWarning:

numpy.ufunc size changed, may indicate binary incompatibility. Expected 216 from C header, got 232 from PyObject



In [40]:
# Define the objective function for Optuna
def objective(trial):

    # Suggest a classifier
    classifier_name = trial.suggest_categorical('classifier', [
        'RandomForestClassifier',
        'GradientBoostingClassifier',
        'SVC',
        'LogisticRegression',
        'DecisionTreeClassifier',
        'KNeighborsClassifier',
        'XGBClassifier',
        'LGBMClassifier',
        'CatBoostClassifier'
    ])
    # Suggest hyperparameters based on the classifier

    if classifier_name == 'RandomForestClassifier':
        n_estimators = trial.suggest_int('n_estimators', 50, 200)
        max_depth = trial.suggest_int('max_depth', 1, 50)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
        model = RandomForestClassifier(n_estimators=n_estimators, max_depth=max_depth,
                                       min_samples_split=min_samples_split, random_state=42)
    elif classifier_name == 'GradientBoostingClassifier':
        n_estimators = trial.suggest_int('n_estimators', 50, 200)
        max_depth = trial.suggest_int('max_depth', 1, 50)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3)
        model = GradientBoostingClassifier(n_estimators=n_estimators, max_depth=max_depth,
                                           learning_rate=learning_rate, random_state=42)
    elif classifier_name == 'SVC':
        C = trial.suggest_float('C', 1e-5, 1e5, log=True)
        kernel = trial.suggest_categorical('kernel', ['linear', 'poly', 'rbf', 'sigmoid'])
        gamma = trial.suggest_categorical('gamma', ['scale', 'auto'])
        model = SVC(C=C, kernel=kernel, gamma=gamma, random_state=42)
    elif classifier_name == 'LogisticRegression':
        C = trial.suggest_float('C', 1e-5, 1e5, log=True)
        solver = trial.suggest_categorical('solver', ['liblinear', 'saga'])
        model = LogisticRegression(C=C, solver=solver, random_state=42)
    elif classifier_name == 'DecisionTreeClassifier':
        max_depth = trial.suggest_int('max_depth', 1, 50)
        min_samples_split = trial.suggest_int('min_samples_split', 2, 20)
        model = DecisionTreeClassifier(max_depth=max_depth, min_samples_split=min_samples_split,
                                        random_state=42)
    elif classifier_name == 'KNeighborsClassifier':
        n_neighbors = trial.suggest_int('n_neighbors', 1, 50)
        weights = trial.suggest_categorical('weights', ['uniform', 'distance'])
        model = KNeighborsClassifier(n_neighbors=n_neighbors, weights=weights)
    elif classifier_name == 'XGBClassifier':
        n_estimators = trial.suggest_int('n_estimators', 50, 200)
        max_depth = trial.suggest_int('max_depth', 1, 50)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3)
        model = XGBClassifier(n_estimators=n_estimators, max_depth=max_depth,
                              learning_rate=learning_rate, random_state=42)
    elif classifier_name == 'LGBMClassifier':
        n_estimators = trial.suggest_int('n_estimators', 50, 200)
        max_depth = trial.suggest_int('max_depth', 1, 50)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3)
        model = LGBMClassifier(n_estimators=n_estimators, max_depth=max_depth,
                               learning_rate=learning_rate, random_state=42)
    elif classifier_name == 'CatBoostClassifier':
        n_estimators = trial.suggest_int('n_estimators', 50, 200)
        depth = trial.suggest_int('depth', 1, 16)
        learning_rate = trial.suggest_float('learning_rate', 0.01, 0.3)
        model = CatBoostClassifier(n_estimators=n_estimators, depth=depth,
                                   learning_rate=learning_rate, verbose=0, random_state=42)
    else:
        raise ValueError(f"Unknown classifier: {classifier_name}")

    # Perform cross-validation and return the mean accuracy
    score = cross_val_score(model, X_train, y_train, cv=5, scoring='accuracy')
    return score.mean()

In [42]:
# Create a study object and optimize the objective function
study = optuna.create_study(direction='maximize', sampler=optuna.samplers.TPESampler())
# Optimize the objective function
study.optimize(objective, n_trials=5)

[I 2025-03-30 03:19:57,958] A new study created in memory with name: no-name-16fb8d75-f23a-4d29-bd30-c65acc1704cf
[I 2025-03-30 03:20:16,774] Trial 0 finished with value: 0.760629081700653 and parameters: {'classifier': 'CatBoostClassifier', 'n_estimators': 66, 'depth': 13, 'learning_rate': 0.15070494931494266}. Best is trial 0 with value: 0.760629081700653.
[I 2025-03-30 03:20:16,996] Trial 1 finished with value: 0.7557110489137677 and parameters: {'classifier': 'LGBMClassifier', 'n_estimators': 125, 'max_depth': 8, 'learning_rate': 0.07821509944724654}. Best is trial 0 with value: 0.760629081700653.


[LightGBM] [Info] Number of positive: 171, number of negative: 320
[LightGBM] [Info] Auto-choosing col-wise multi-threading, the overhead of testing was 0.000367 seconds.
You can set `force_col_wise=true` to remove the overhead.
[LightGBM] [Info] Total Bins 583
[LightGBM] [Info] Number of data points in the train set: 491, number of used features: 8
[LightGBM] [Info] [binary:BoostFromScore]: pavg=0.348269 -> initscore=-0.626657
[LightGBM] [Info] Start training from score -0.626657
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best gain: -inf
[LightGBM] [Warning] No further splits with positive gain, best 

[I 2025-03-30 03:20:17,035] Trial 2 finished with value: 0.7214714114354258 and parameters: {'classifier': 'DecisionTreeClassifier', 'max_depth': 6, 'min_samples_split': 12}. Best is trial 0 with value: 0.760629081700653.
[I 2025-03-30 03:20:18,540] Trial 3 finished with value: 0.7328535252565641 and parameters: {'classifier': 'XGBClassifier', 'n_estimators': 67, 'max_depth': 41, 'learning_rate': 0.25090403558981356}. Best is trial 0 with value: 0.760629081700653.
[I 2025-03-30 03:20:18,577] Trial 4 finished with value: 0.7491803278688525 and parameters: {'classifier': 'KNeighborsClassifier', 'n_neighbors': 46, 'weights': 'distance'}. Best is trial 0 with value: 0.760629081700653.


In [43]:
# Retrieve the best trial
best_trial = study.best_trial
print("Best trial parameters:", best_trial.params)
print("Best trial accuracy:", best_trial.value)

Best trial parameters: {'classifier': 'CatBoostClassifier', 'n_estimators': 66, 'depth': 13, 'learning_rate': 0.15070494931494266}
Best trial accuracy: 0.760629081700653


In [44]:
study.trials_dataframe()

,number,value,datetime_start,datetime_complete,duration,params_classifier,params_depth,params_learning_rate,params_max_depth,params_min_samples_split,params_n_estimators,params_n_neighbors,params_weights,state
0,0,0.760629,2025-03-30 03:19:57.960514,2025-03-30 03:20:16.774393,0 days 00:00:18.813879,CatBoostClassifier,13.0,0.150705,NaN,NaN,66.0,NaN,NaN,COMPLETE
1,1,0.755711,2025-03-30 03:20:16.775224,2025-03-30 03:20:16.996053,0 days 00:00:00.220829,LGBMClassifier,NaN,0.078215,8.0,NaN,125.0,NaN,NaN,COMPLETE
2,2,0.721471,2025-03-30 03:20:16.997252,2025-03-30 03:20:17.035241,0 days 00:00:00.037989,DecisionTreeClassifier,NaN,NaN,6.0,12.0,NaN,NaN,NaN,COMPLETE
3,3,0.732854,2025-03-30 03:20:17.036996,2025-03-30 03:20:18.540585,0 days 00:00:01.503589,XGBClassifier,NaN,0.250904,41.0,NaN,67.0,NaN,NaN,COMPLETE
4,4,0.749180,2025-03-30 03:20:18.541461,2025-03-30 03:20:18.577599,0 days 00:00:00.036138,KNeighborsClassifier,NaN,NaN,NaN,NaN,NaN,46.0,distance,COMPLETE


In [45]:
study.trials_dataframe()['params_classifier'].value_counts()

params_classifier
CatBoostClassifier        1
LGBMClassifier            1
DecisionTreeClassifier    1
XGBClassifier             1
KNeighborsClassifier      1
Name: count, dtype: int64

In [46]:
study.trials_dataframe().groupby('params_classifier')['value'].mean()

params_classifier
CatBoostClassifier        0.760629
DecisionTreeClassifier    0.721471
KNeighborsClassifier      0.749180
LGBMClassifier            0.755711
XGBClassifier             0.732854
Name: value, dtype: float64